# Mediterranean Sea Surface Salinity — Climatology, Anomalies & Extremes (1987–2025)

**Workflow**
1. Open the Mediterranean Sea Physics Reanalysis (`cmems_mod_med_phy-sal_my_4.2km_P1M-m`) — monthly means, lazy (dask).
2. Compute the **yearly mean salinity** at every pixel and explore it with a year slider.
3. Compute the **long-term climatological mean** at every pixel (mean over 1987–2025).
4. Compute the **monthly salinity anomaly** = monthly value − pixel climatological mean.
5. Visualise the monthly anomaly with an interactive time slider showing the current month.
6. At every pixel, find the **minimum** and **maximum** of the anomaly time-series.
7. Plot spatial maps of the min-anomaly and max-anomaly fields.

> **Credentials**: requires a `~/.netrc` entry for `auth.marine.copernicus.eu`.

In [ ]:
import cmocean
import copernicusmarine
import hvplot.xarray
import numpy as np
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Mediterranean bounding box
MED_BBOX = dict(
    minimum_longitude=-6,
    maximum_longitude=42,
    minimum_latitude=30,
    maximum_latitude=47,
)

# The reanalysis starts January 1987; request through end of 2025
TIME_START = "1987-01-01"
TIME_END   = "2025-12-31"

## 1 — Open the salinity dataset (lazy)

In [ ]:
%%time
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-sal_my_4.2km_P1M-m",
    start_datetime=TIME_START,
    end_datetime=TIME_END,
    **MED_BBOX,
)
ds

In [ ]:
# Surface salinity: select depth nearest to 0 m — result is (time, latitude, longitude), still lazy
sal = ds["so"].sel(depth=0, method="nearest")
print(f"Salinity shape: {sal.dims} = {dict(zip(sal.dims, sal.shape))}")
sal

## 2 — Yearly mean salinity with time slider

Resample monthly → annual, compute (≈ 39 frames × 380 × 1016 ≈ 60 MB), then display with a year scrubber.

In [ ]:
%%time
sal_yearly = sal.resample(time="YE").mean().compute()

# Replace the datetime coordinate with plain integer years for a cleaner slider
years = sal_yearly.time.dt.year.values
sal_yearly = (
    sal_yearly
    .assign_coords(year=("time", years))
    .swap_dims({"time": "year"})
    .drop_vars("time")
)
sal_yearly.attrs["long_name"] = "Annual Mean Salinity"
sal_yearly.attrs["units"] = "PSU"
print(f"Yearly salinity shape: {sal_yearly.dims} = {dict(zip(sal_yearly.dims, sal_yearly.shape))}")
sal_yearly

In [ ]:
sal_yearly.hvplot(
    x="longitude",
    y="latitude",
    groupby="year",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.haline,
    clim=(float(sal_yearly.min()), float(sal_yearly.max())),
    clabel="Salinity (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title="Mediterranean Annual Mean Salinity — use slider to step through years",
    fontscale=1.2,
    widget_type="scrubber",
    widget_location="bottom",
)

## 3 — Long-term climatological mean

Average every pixel over the full record (1987–2025). This is the reference field for anomaly computation.

In [ ]:
%%time
sal_clim = sal.mean(dim="time").compute()
sal_clim.attrs["long_name"] = f"Climatological Mean Salinity ({TIME_START[:4]}\u2013{TIME_END[:4]})"
sal_clim.attrs["units"] = "PSU"
print(f"Salinity clim range: {float(sal_clim.min()):.2f} \u2013 {float(sal_clim.max()):.2f} PSU")
sal_clim

In [ ]:
sal_clim.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap=cmocean.cm.haline,
    clabel="Salinity (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Climatological Mean Salinity ({TIME_START[:4]}–{TIME_END[:4]})",
    fontscale=1.2,
)

## 4 — Monthly salinity anomaly

Anomaly = monthly salinity − pixel climatological mean.

The subtraction broadcasts `sal_clim` (2-D) over the time axis of `sal` (3-D). The result remains a lazy dask array — no data is loaded into RAM until we ask for an explicit reduction.

In [ ]:
sal_anom = sal - sal_clim
sal_anom.attrs["long_name"] = "Salinity Anomaly"
sal_anom.attrs["units"] = "PSU"
print(f"Anomaly shape: {sal_anom.dims} = {dict(zip(sal_anom.dims, sal_anom.shape))}")
sal_anom

## 4b — Monthly salinity anomaly map with time slider

In [ ]:
import pandas as pd
import panel as pn

pn.extension()

# Load the full monthly anomaly into memory — needed for a responsive time slider
sal_anom_loaded = sal_anom.compute()

# Symmetric color limits so blue/red are balanced around zero
anom_max = float(max(abs(sal_anom_loaded.min()), abs(sal_anom_loaded.max())))
times = sal_anom_loaded.time.values

# Integer-index Player widget (play/pause/step controls + scrubber)
player = pn.widgets.Player(
    start=0, end=len(times) - 1, value=0,
    loop_policy="loop", width=800, name="",
)

def plot_sal_frame(idx):
    label = pd.Timestamp(times[idx]).strftime("%B %Y")
    return sal_anom_loaded.isel(time=idx).hvplot(
        x="longitude",
        y="latitude",
        rasterize=True,
        geo=True,
        cmap="RdBu_r",
        clim=(-anom_max, anom_max),
        clabel="Salinity Anomaly (PSU)",
        tiles="OSM",
        frame_width=800,
        frame_height=450,
        title=f"Mediterranean Monthly Salinity Anomaly \u2014 {label}",
        fontscale=1.2,
    )

pn.Column(pn.bind(plot_sal_frame, player), player)

## 5 — Pixel-wise minimum and maximum anomaly

Collapse the time dimension by taking `min` and `max`. Both reductions are fused into a single dask graph pass over the data.

In [ ]:
%%time
import dask

# Compute both reductions in one scheduler call to share the I/O
sal_anom_min, sal_anom_max = dask.compute(
    sal_anom.min(dim="time"),
    sal_anom.max(dim="time"),
)

sal_anom_min.attrs["long_name"] = "Minimum Salinity Anomaly"
sal_anom_min.attrs["units"] = "PSU"
sal_anom_max.attrs["long_name"] = "Maximum Salinity Anomaly"
sal_anom_max.attrs["units"] = "PSU"

print(f"Min anomaly range : {float(sal_anom_min.min()):.3f} \u2013 {float(sal_anom_min.max()):.3f} PSU")
print(f"Max anomaly range : {float(sal_anom_max.min()):.3f} \u2013 {float(sal_anom_max.max()):.3f} PSU")

## 6 — Spatial maps of min and max anomaly

A diverging colormap (`RdBu_r`) centred on zero makes fresher/saltier departures immediately readable.

In [ ]:
# Shared symmetric color limits so both maps are directly comparable
anom_abs_max = max(
    abs(float(sal_anom_min.min())),
    abs(float(sal_anom_max.max())),
)
clim_anom = (-anom_abs_max, anom_abs_max)
print(f"Symmetric color range: {clim_anom[0]:.3f} \u2013 {clim_anom[1]:.3f} PSU")

In [ ]:
plot_min = sal_anom_min.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Minimum Salinity Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_min

In [ ]:
plot_max = sal_anom_max.hvplot(
    x="longitude",
    y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdBu_r",
    clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=800,
    frame_height=450,
    title=f"Mediterranean Maximum Salinity Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.2,
)
plot_max

## 7 — Side-by-side comparison

In [ ]:
plot_min_small = sal_anom_min.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Min Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

plot_max_small = sal_anom_max.hvplot(
    x="longitude", y="latitude",
    rasterize=True, geo=True,
    cmap="RdBu_r", clim=clim_anom,
    clabel="Salinity Anomaly (PSU)",
    tiles="OSM",
    frame_width=560, frame_height=350,
    title=f"Max Anomaly ({TIME_START[:4]}\u2013{TIME_END[:4]})",
    fontscale=1.1,
)

(plot_min_small + plot_max_small).cols(2)

## 8 — Export annual mean salinity animation as MP4

Render one frame per year (1987–2025) of the **Mediterranean annual mean salinity** using matplotlib and save it as a downloadable MP4 video. Requires `ffmpeg` to be available on the system.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import numpy as np
import cmocean

VIDEO_PATH = "med_climatological_mean_salinity.mp4"

# sal_yearly is already computed above (year × lat × lon)
lons = sal_yearly.longitude.values
lats = sal_yearly.latitude.values
years_list = sal_yearly.year.values

vmin = float(np.nanmin(sal_yearly.values))
vmax = float(np.nanmax(sal_yearly.values))

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_facecolor("#c8e6f5")

# First frame
data0 = sal_yearly.sel(year=years_list[0]).values
img = ax.pcolormesh(
    lons, lats, data0,
    cmap=cmocean.cm.haline,
    vmin=vmin, vmax=vmax,
    shading="auto",
)
cb = fig.colorbar(img, ax=ax, label="Salinity (PSU)", fraction=0.03, pad=0.02)
title = ax.set_title(
    f"Mediterranean Annual Mean Salinity — {years_list[0]}",
    fontsize=14,
)
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.tight_layout()

def update(frame_idx):
    yr = years_list[frame_idx]
    data = sal_yearly.sel(year=yr).values
    img.set_array(data.ravel())
    title.set_text(f"Mediterranean Annual Mean Salinity — {yr}")
    return img, title

ani = animation.FuncAnimation(
    fig,
    update,
    frames=len(years_list),
    interval=400,     # ms between frames
    blit=False,
)

ani.save(VIDEO_PATH, writer="ffmpeg", fps=3, dpi=120)
plt.close(fig)
print(f"Video saved → {VIDEO_PATH}")

In [ ]:
from IPython.display import FileLink, display
display(FileLink(VIDEO_PATH, result_html_prefix="Download video: "))